# Proving RNG implementation equality

Prove that `Taus88RNG` from the Boost library and `ExtractedRNG` from the reverse-engineered game code produce the same state transition.



In [2]:
from register import SymbolicRegister
from rngs import ExtractedRNG, Taus88RNG

ModuleNotFoundError: No module named 'register'

To see the respective implementations of the RNGs see [rngs.py](./rngs.py) and [register.py](./register.py).

First we will initialize both the `Taus88RNG` and `ExtractedRNG` reimplementations with the same symbolic state.
Note that with this reimplementation we can use a `SymbolicRegister` which represents register states using symbolic variables rather than concrete values. This allows us to prove the equality of the two RNGs for all possible initial states rather than just a single one.

In [ ]:
# Declare the 2 rngs with symbolic registers (a GF(2) variable per bit)
rng_taus88 = Taus88RNG(
    (
        SymbolicRegister(32, "a"),
        SymbolicRegister(32, "b"),
        SymbolicRegister(32, "c"),
    )
)
rng_extracted = ExtractedRNG(
    (
        SymbolicRegister(32, "a"),
        SymbolicRegister(32, "b"),
        SymbolicRegister(32, "c"),
    )
)


Let's take a look at the initial state of both RNGs:

In [ ]:
# Print the state of both RNGs directly after initialization, to verify that they are the same.
def print_rng_states(rng):
    """Print the state of the given RNG."""
    for i, register in enumerate(rng.state):
        print(f"  Register {i}: {register}")


print("RNG 1 (taus88):")
print_rng_states(rng_taus88)
print()
print("RNG 2 (extracted):")
print_rng_states(rng_extracted)

RNG 1 (taus88):
  Register 0: r[32] = [a31, a30, a29, a28, a27, a26, a25, a24, a23, a22, a21, a20, a19, a18, a17, a16, a15, a14, a13, a12, a11, a10, a9, a8, a7, a6, a5, a4, a3, a2, a1, a0]
  Register 1: r[32] = [b31, b30, b29, b28, b27, b26, b25, b24, b23, b22, b21, b20, b19, b18, b17, b16, b15, b14, b13, b12, b11, b10, b9, b8, b7, b6, b5, b4, b3, b2, b1, b0]
  Register 2: r[32] = [c31, c30, c29, c28, c27, c26, c25, c24, c23, c22, c21, c20, c19, c18, c17, c16, c15, c14, c13, c12, c11, c10, c9, c8, c7, c6, c5, c4, c3, c2, c1, c0]

RNG 2 (extracted):
  Register 0: r[32] = [a31, a30, a29, a28, a27, a26, a25, a24, a23, a22, a21, a20, a19, a18, a17, a16, a15, a14, a13, a12, a11, a10, a9, a8, a7, a6, a5, a4, a3, a2, a1, a0]
  Register 1: r[32] = [b31, b30, b29, b28, b27, b26, b25, b24, b23, b22, b21, b20, b19, b18, b17, b16, b15, b14, b13, b12, b11, b10, b9, b8, b7, b6, b5, b4, b3, b2, b1, b0]
  Register 2: r[32] = [c31, c30, c29, c28, c27, c26, c25, c24, c23, c22, c21, c20, c19, c18, c17, c

Seeing that both RNGs are initialized with the same (symbolic) state, we step both RNGs forward a single step and compare their states again. If they are equal, this means that each step executes the same state transition and thus both RNGs will produce the same output sequence for any given initial state.

In [ ]:
# Now step both RNGs forward a single step
rng_taus88.step()
rng_extracted.step()

print("After stepping both RNGs forward a single step:")
print_rng_states(rng_taus88)
print()
print_rng_states(rng_extracted)


After stepping both RNGs forward a single step:
  Register 0: r[32] = [a19, a18, a17, a16, a15, a14, a13, a12, a11, a10, a9, a8, a7, a6, a5, a4, a3, a2, a1, a18 ^ a31, a17 ^ a30, a16 ^ a29, a15 ^ a28, a14 ^ a27, a13 ^ a26, a12 ^ a25, a11 ^ a24, a10 ^ a23, a22 ^ a9, a21 ^ a8, a20 ^ a7, a19 ^ a6]
  Register 1: r[32] = [b27, b26, b25, b24, b23, b22, b21, b20, b19, b18, b17, b16, b15, b14, b13, b12, b11, b10, b9, b8, b7, b6, b5, b4, b3, b29 ^ b31, b28 ^ b30, b27 ^ b29, b26 ^ b28, b25 ^ b27, b24 ^ b26, b23 ^ b25]
  Register 2: r[32] = [c14, c13, c12, c11, c10, c9, c8, c7, c6, c5, c4, c28 ^ c31, c27 ^ c30, c26 ^ c29, c25 ^ c28, c24 ^ c27, c23 ^ c26, c22 ^ c25, c21 ^ c24, c20 ^ c23, c19 ^ c22, c18 ^ c21, c17 ^ c20, c16 ^ c19, c15 ^ c18, c14 ^ c17, c13 ^ c16, c12 ^ c15, c11 ^ c14, c10 ^ c13, c12 ^ c9, c11 ^ c8]

  Register 0: r[32] = [a19, a18, a17, a16, a15, a14, a13, a12, a11, a10, a9, a8, a7, a6, a5, a4, a3, a2, a1, a18 ^ a31, a17 ^ a30, a16 ^ a29, a15 ^ a28, a14 ^ a27, a13 ^ a26, a12 ^ a25

Now rather than manually going through each entry which is error-prone, we can instead also ask sympy to do the work for us.
The following compares all registers of both RNGs after stepping forward a single step.

In [ ]:
for i, (reg1, reg2) in enumerate(zip(rng_taus88.state, rng_extracted.state)):
    if reg1 != reg2:
        print(f"Failed at register {i}: {reg1} != {reg2}")
        break
else:
    print("Success! The RNGs are equal!")

Success! The RNGs are equal!
